<a href="https://colab.research.google.com/github/Deelaksha22/NEXUS-AI-AGENT/blob/main/NEXUS_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Dependency Installation
!pip install -q -U google-genai pydantic tabulate

In [ ]:
# Cell 2: Data Schemas
from pydantic import BaseModel, Field
from typing import List, Optional
from enum import Enum
from datetime import datetime

class PriorityLevel(str, Enum):
    LOW = "LOW"
    MEDIUM = "MEDIUM"
    HIGH = "HIGH"
    CRITICAL = "CRITICAL"

class ExtractedTask(BaseModel):
    title: str = Field(description="Short, clear name of the task")
    description: Optional[str] = Field(default="", description="Relevant context or sub-details")
    deadline: Optional[str] = Field(description="Deadline ISO format YYYY-MM-DD HH:MM, or relative time if date is not known")
    duration_minutes: int = Field(default=60, description="Estimated duration in minutes. Default to 60 if unspecified")
    priority: PriorityLevel = Field(default=PriorityLevel.MEDIUM, description="Task urgency and importance")
    dependencies: List[str] = Field(default_factory=list, description="Titles of tasks that must be finished before this one")

class FixedCommitment(BaseModel):
    title: str = Field(description="Name of the fixed event, e.g., College Classes, Doctors Appointment")
    start_time: str = Field(description="Start time in ISO format YYYY-MM-DD HH:MM or HH:MM")
    end_time: str = Field(description="End time in ISO format YYYY-MM-DD HH:MM or HH:MM")

class ParsedUserInput(BaseModel):
    tasks: List[ExtractedTask] = Field(description="List of flexible actionable tasks")
    fixed_commitments: List[FixedCommitment] = Field(description="Fixed non-negotiable blocks of time")
    wake_up_time: str = Field(default="08:00", description="User's starting hour for the day (HH:MM)")
    sleep_time: str = Field(default="23:00", description="User's end of day cutoff (HH:MM)")
    missing_information_questions: List[str] = Field(
        default_factory=list,
        description="Clarifying questions if a critical deadline or task requirement is ambiguous"
    )

In [ ]:
# Cell 3: Cognitive Input Module using gemini-3.8-flash
import os
import time
from google import genai
from google.genai import types
from google.genai.errors import ServerError, ClientError
from google.colab import userdata

# Initialize GenAI Client
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

# Explicitly set the active model requested by the API
MODEL_ID = "gemini-3.8-flash"

def parse_user_schedule_intent(user_raw_text: str, current_context_time: str) -> ParsedUserInput:
    """
    Translates raw conversational user input into a validated, structured object.
    """
    system_instruction = f"""
    You are the Cognitive Input Engine for NEXUS, an autonomous personal productivity agent.
    Current Date and Time Context: {current_context_time}

    Your job:
    1. Read the user's daily goals, chores, tests, and schedule constraints.
    2. Separate negotiable tasks (assignments, study time) from non-negotiable commitments (classes, meetings, shifts).
    3. Infer reasonable durations if omitted (e.g., a quiz is typically 30-60 mins; studying for a project is 90-120 mins).
    4. Flag missing information if an explicit deadline lacks a date/time.
    5. Always return strictly valid JSON matching the schema.
    """

    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=user_raw_text,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    response_mime_type="application/json",
                    response_schema=ParsedUserInput,
                    temperature=0.1
                )
            )
            return ParsedUserInput.model_validate_json(response.text)

        except ServerError:
            wait_seconds = (attempt + 1) * 2
            print(f"⚠️ Model busy (503). Retrying in {wait_seconds}s...")
            time.sleep(wait_seconds)
        except Exception as e:
            if attempt == max_retries - 1:
                raise e
            print(f"⚠️ Attempt {attempt + 1} encountered an error: {e}. Retrying in 2s...")
            time.sleep(2)

print(f"✅ Parser configured successfully with model: {MODEL_ID}")

✅ Parser configured successfully with model: gemini-3.8-flash


In [ ]:
# Cell 4: Execute Test Input
from datetime import datetime

# Simulating current time context: 2026-10-10 at 07:00 AM
simulated_now = "2026-10-10 07:00"

sample_text = """
I have to complete my Java DSA assignment by 5 PM tomorrow.
I have a DBMS quiz tomorrow morning.
I also need to work on my college project for two hours.
I have classes from 9 AM to 4 PM.
"""

parsed_data = parse_user_schedule_intent(sample_text, simulated_now)

print("--- FIXED COMMITMENTS ---")
for fc in parsed_data.fixed_commitments:
    print(f"• {fc.title}: {fc.start_time} to {fc.end_time}")

print("\n--- EXTRACTED FLEXIBLE TASKS ---")
for t in parsed_data.tasks:
    print(f"• [{t.priority}] {t.title} | Duration: {t.duration_minutes}m | Deadline: {t.deadline}")

if parsed_data.missing_information_questions:
    print("\n--- CLARIFICATION QUESTIONS ---")
    for q in parsed_data.missing_information_questions:
        print(f"? {q}")

--- FIXED COMMITMENTS ---
• College Classes: 2026-10-10 09:00 to 2026-10-10 16:00

--- EXTRACTED FLEXIBLE TASKS ---
• [PriorityLevel.HIGH] Complete Java DSA Assignment | Duration: 120m | Deadline: 2026-10-11 17:00
• [PriorityLevel.HIGH] Prepare for DBMS Quiz | Duration: 60m | Deadline: 2026-10-11 09:00
• [PriorityLevel.MEDIUM] Work on College Project | Duration: 120m | Deadline: None

--- CLARIFICATION QUESTIONS ---
? What is the exact time and duration for the DBMS quiz tomorrow morning?
? Are the 9 AM to 4 PM classes scheduled for today, tomorrow, or both days?
? Is there a specific deadline for the 2-hour college project work?


In [ ]:
# Cell 5: Scheduling Engine and Conflict Detector
from datetime import datetime, timedelta
from typing import List, Dict, Any

class ScheduledSlot(BaseModel):
    title: str
    slot_type: str  # "TASK", "FIXED", or "BREAK"
    start_time: str
    end_time: str
    duration_minutes: int
    details: Optional[str] = ""

class ScheduleResult(BaseModel):
    schedule: List[ScheduledSlot]
    conflicts: List[str]
    next_action_recommendation: Optional[str]
    total_study_minutes: int

def parse_time_helper(time_str: str, base_date: datetime) -> datetime:
    """
    Parses either 'YYYY-MM-DD HH:MM' or simple 'HH:MM' strings into full datetime objects.
    """
    try:
        if len(time_str) == 5 and ":" in time_str:  # Format: "09:00"
            hour, minute = map(int, time_str.split(":"))
            return base_date.replace(hour=hour, minute=minute, second=0, microsecond=0)
        else:
            return datetime.fromisoformat(time_str)
    except Exception:
        # Fallback if format doesn't match
        return base_date + timedelta(hours=1)

def generate_optimized_schedule(parsed_input: ParsedUserInput, current_date_str: str) -> ScheduleResult:
    base_date = datetime.strptime(current_date_str, "%Y-%m-%d %H:%M")

    # 1. Establish the Day's Boundaries
    day_start = parse_time_helper(parsed_input.wake_up_time, base_date)
    day_end = parse_time_helper(parsed_input.sleep_time, base_date)

    # 2. Parse and Sort Fixed Commitments
    fixed_blocks = []
    for fc in parsed_input.fixed_commitments:
        f_start = parse_time_helper(fc.start_time, base_date)
        f_end = parse_time_helper(fc.end_time, base_date)
        fixed_blocks.append({
            "title": fc.title,
            "start": f_start,
            "end": f_end
        })
    fixed_blocks.sort(key=lambda x: x["start"])

    # 3. Calculate Free Time Intervals between Fixed Commitments
    free_intervals = []
    cursor = day_start

    for block in fixed_blocks:
        if block["start"] > cursor:
            free_intervals.append({"start": cursor, "end": block["start"]})
        cursor = max(cursor, block["end"])

    if cursor < day_end:
        free_intervals.append({"start": cursor, "end": day_end})

    # 4. Sort Flexible Tasks by Priority & Deadlines
    priority_weights = {
        PriorityLevel.CRITICAL: 4,
        PriorityLevel.HIGH: 3,
        PriorityLevel.MEDIUM: 2,
        PriorityLevel.LOW: 1
    }

    def task_sort_key(task: ExtractedTask):
        # Earliest deadline gets top priority
        dl_score = 999999
        if task.deadline:
            try:
                dl_dt = parse_time_helper(task.deadline, base_date)
                dl_score = (dl_dt - base_date).total_seconds()
            except Exception:
                pass
        # Negative weight so higher priority comes first
        return (dl_score, -priority_weights.get(task.priority, 1))

    sorted_tasks = sorted(parsed_input.tasks, key=task_sort_key)

    # 5. Fit Tasks into Free Intervals
    final_timeline: List[ScheduledSlot] = []
    conflicts: List[str] = []
    total_study_minutes = 0

    # Add fixed commitments into the timeline right away
    for b in fixed_blocks:
        duration = int((b["end"] - b["start"]).total_seconds() // 60)
        final_timeline.append(ScheduledSlot(
            title=b["title"],
            slot_type="FIXED",
            start_time=b["start"].strftime("%H:%M"),
            end_time=b["end"].strftime("%H:%M"),
            duration_minutes=duration,
            details="Fixed Commitment (Non-negotiable)"
        ))

    # Place tasks one by one
    for task in sorted_tasks:
        remaining_duration = task.duration_minutes
        placed = False

        for interval in free_intervals:
            interval_duration = int((interval["end"] - interval["start"]).total_seconds() // 60)

            if interval_duration >= remaining_duration:
                task_start = interval["start"]
                task_end = task_start + timedelta(minutes=remaining_duration)

                # Check deadline conflict
                if task.deadline:
                    try:
                        dl_dt = parse_time_helper(task.deadline, base_date)
                        if task_end > dl_dt:
                            conflicts.append(
                                f"CONFLICT: '{task.title}' finishes at {task_end.strftime('%H:%M')}, which misses the deadline {task.deadline}!"
                            )
                    except Exception:
                        pass

                final_timeline.append(ScheduledSlot(
                    title=task.title,
                    slot_type="TASK",
                    start_time=task_start.strftime("%H:%M"),
                    end_time=task_end.strftime("%H:%M"),
                    duration_minutes=remaining_duration,
                    details=f"Priority: {task.priority.value}"
                ))
                total_study_minutes += remaining_duration

                # Automatically add a 15-minute rest break after any task >= 45 minutes
                break_duration = 15 if remaining_duration >= 45 else 0
                new_start = task_end + timedelta(minutes=break_duration)

                if break_duration > 0 and new_start < interval["end"]:
                    final_timeline.append(ScheduledSlot(
                        title="Rest & Re-energize",
                        slot_type="BREAK",
                        start_time=task_end.strftime("%H:%M"),
                        end_time=new_start.strftime("%H:%M"),
                        duration_minutes=break_duration,
                        details="Short mental recovery break"
                    ))

                # Update the remaining window
                interval["start"] = new_start
                placed = True
                break

        if not placed:
            conflicts.append(f"UNSCHEDULED: '{task.title}' ({task.duration_minutes}m) could not fit into remaining free hours.")

    # Sort the full timeline chronologically
    final_timeline.sort(key=lambda s: s.start_time)

    # Determine immediate next actionable task
    next_task = next((s for s in final_timeline if s.slot_type == "TASK"), None)
    recommendation = (
        f"Start with '{next_task.title}' at {next_task.start_time}."
        if next_task else "No immediate pending tasks for today!"
    )

    return ScheduleResult(
        schedule=final_timeline,
        conflicts=conflicts,
        next_action_recommendation=recommendation,
        total_study_minutes=total_study_minutes
    )

In [ ]:
# Cell 6: Run and Display Schedule
from tabulate import tabulate

# Pass the output from Cell 4 into our scheduling engine
schedule_output = generate_optimized_schedule(parsed_data, simulated_now)

# Prepare clean table
table_rows = []
for slot in schedule_output.schedule:
    prefix = "📅" if slot.slot_type == "FIXED" else ("☕" if slot.slot_type == "BREAK" else "⚡")
    table_rows.append([
        f"{prefix} {slot.title}",
        slot.start_time,
        slot.end_time,
        f"{slot.duration_minutes}m",
        slot.details
    ])

print("="*65)
print("                NEXUS DAILY MASTER SCHEDULE")
print("="*65)
print(tabulate(table_rows, headers=["Activity / Task", "Start", "End", "Duration", "Details"], tablefmt="fancy_grid"))

print(f"\n🎯 NEXT ACTION: {schedule_output.next_action_recommendation}")
print(f"⏱️ TOTAL FOCUSED WORK: {schedule_output.total_study_minutes // 60}h {schedule_output.total_study_minutes % 60}m")

if schedule_output.conflicts:
    print("\n⚠️ WARNINGS & CONFLICTS DETECTED:")
    for c in schedule_output.conflicts:
        print(f"• {c}")
else:
    print("\n✅ All tasks fit with zero scheduling conflicts!")

                NEXUS DAILY MASTER SCHEDULE
╒═════════════════════════════════╤═════════╤═══════╤════════════╤═══════════════════════════════════╕
│ Activity / Task                 │ Start   │ End   │ Duration   │ Details                           │
╞═════════════════════════════════╪═════════╪═══════╪════════════╪═══════════════════════════════════╡
│ ⚡ Prepare for DBMS Quiz        │ 07:00   │ 08:00 │ 60m        │ Priority: HIGH                    │
├─────────────────────────────────┼─────────┼───────┼────────────┼───────────────────────────────────┤
│ ☕ Rest & Re-energize           │ 08:00   │ 08:15 │ 15m        │ Short mental recovery break       │
├─────────────────────────────────┼─────────┼───────┼────────────┼───────────────────────────────────┤
│ 📅 College Classes              │ 09:00   │ 16:00 │ 420m       │ Fixed Commitment (Non-negotiable) │
├─────────────────────────────────┼─────────┼───────┼────────────┼───────────────────────────────────┤
│ ⚡ Complete Java DSA Assignment

In [ ]:
# Cell 7: SQLite Database Manager
import sqlite3
import pandas as pd
from typing import List, Optional

DB_FILE = "nexus_tasks.db"

def init_db():
    """Initializes the database schema if it doesn't already exist."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    # Table 1: Tasks Table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS tasks (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT NOT NULL,
        description TEXT,
        deadline TEXT,
        duration_minutes INTEGER,
        priority TEXT,
        status TEXT DEFAULT 'PENDING',
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)

    # Table 2: Daily Schedule History
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS schedules (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        date_str TEXT,
        slot_type TEXT,
        title TEXT,
        start_time TEXT,
        end_time TEXT,
        duration_minutes INTEGER,
        details TEXT
    )
    """)
    conn.commit()
    conn.close()

def save_tasks_to_db(tasks: List[ExtractedTask]):
    """Stores newly parsed tasks into the database."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    for task in tasks:
        cursor.execute("""
        INSERT INTO tasks (title, description, deadline, duration_minutes, priority, status)
        VALUES (?, ?, ?, ?, ?, 'PENDING')
        """, (task.title, task.description, task.deadline, task.duration_minutes, task.priority.value))
    conn.commit()
    conn.close()

def save_schedule_to_db(schedule: List[ScheduledSlot], date_str: str):
    """Saves the generated daily schedule slots."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    # Clear existing schedule for this date to avoid duplicate runs
    cursor.execute("DELETE FROM schedules WHERE date_str = ?", (date_str,))
    for slot in schedule:
        cursor.execute("""
        INSERT INTO schedules (date_str, slot_type, title, start_time, end_time, duration_minutes, details)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (date_str, slot.slot_type, slot.title, slot.start_time, slot.end_time, slot.duration_minutes, slot.details))
    conn.commit()
    conn.close()

def update_task_status(task_id: int, new_status: str):
    """Updates a task status: PENDING, IN_PROGRESS, or COMPLETED."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("UPDATE tasks SET status = ? WHERE id = ?", (new_status.upper(), task_id))
    conn.commit()
    conn.close()

def get_tasks_as_dataframe() -> pd.DataFrame:
    """Reads tasks from SQLite into a Pandas DataFrame for inspection."""
    conn = sqlite3.connect(DB_FILE)
    df = pd.read_sql_query("SELECT id, title, priority, duration_minutes, deadline, status FROM tasks", conn)
    conn.close()
    return df

# Initialize database
init_db()
print("✅ SQLite database initialized successfully!")

✅ SQLite database initialized successfully!


In [ ]:
# Cell 8: Email Notification Module
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from google.colab import userdata

def send_nexus_schedule_email(
    recipient_email: str,
    schedule_res: ScheduleResult,
    date_context: str
) -> bool:
    """
    Constructs an HTML email containing the generated schedule and next action,
    then sends it via SMTP.
    """
    try:
        sender_email = userdata.get('SENDER_EMAIL')
        sender_password = userdata.get('SENDER_APP_PASSWORD')
    except Exception:
        print("⚠️ SENDER_EMAIL or SENDER_APP_PASSWORD not configured in Colab Secrets.")
        return False

    # 1. Build HTML Table Rows
    rows_html = ""
    for slot in schedule_res.schedule:
        bg_color = "#f4f6f9"
        badge_color = "#3498db"
        if slot.slot_type == "FIXED":
            badge_color = "#e67e22"
            bg_color = "#fdfaf6"
        elif slot.slot_type == "BREAK":
            badge_color = "#2ecc71"
            bg_color = "#f4fbf7"

        rows_html += f"""
        <tr style="background-color: {bg_color}; border-bottom: 1px solid #e0e0e0;">
            <td style="padding: 10px; font-weight: bold;">{slot.start_time} - {slot.end_time}</td>
            <td style="padding: 10px;">
                <span style="background: {badge_color}; color: white; padding: 2px 8px; border-radius: 4px; font-size: 11px;">
                    {slot.slot_type}
                </span>
                <strong style="margin-left: 8px;">{slot.title}</strong>
            </td>
            <td style="padding: 10px;">{slot.duration_minutes}m</td>
            <td style="padding: 10px; font-size: 12px; color: #555;">{slot.details}</td>
        </tr>
        """

    # 2. Build Conflict Warning Block
    conflict_html = ""
    if schedule_res.conflicts:
        conflict_list = "".join([f"<li>{c}</li>" for c in schedule_res.conflicts])
        conflict_html = f"""
        <div style="background-color: #fee; border-left: 4px solid #e74c3c; padding: 12px; margin-top: 15px;">
            <strong style="color: #c0392b;">⚠️ Scheduling Warnings:</strong>
            <ul style="margin: 5px 0 0 15px; color: #555;">{conflict_list}</ul>
        </div>
        """

    # 3. Assemble Complete HTML Template
    html_content = f"""
    <html>
    <body style="font-family: Arial, sans-serif; color: #333; line-height: 1.6; max-width: 650px; margin: auto; padding: 20px;">
        <div style="border-bottom: 2px solid #2c3e50; padding-bottom: 10px; margin-bottom: 20px;">
            <h2 style="color: #2c3e50; margin: 0;">⚡ NEXUS Daily Briefing</h2>
            <small style="color: #7f8c8d;">Plan for: {date_context}</small>
        </div>

        <div style="background: #e8f4fd; border-radius: 6px; padding: 15px; margin-bottom: 20px;">
            <h3 style="margin: 0 0 5px 0; color: #2980b9;">🎯 Immediate Next Action:</h3>
            <p style="margin: 0; font-size: 16px; font-weight: bold; color: #1f618d;">
                {schedule_res.next_action_recommendation}
            </p>
        </div>

        <h3>📋 Your Daily Schedule</h3>
        <table style="width: 100%; border-collapse: collapse; text-align: left; font-size: 14px;">
            <thead>
                <tr style="background-color: #2c3e50; color: white;">
                    <th style="padding: 10px;">Time</th>
                    <th style="padding: 10px;">Activity</th>
                    <th style="padding: 10px;">Duration</th>
                    <th style="padding: 10px;">Notes</th>
                </tr>
            </thead>
            <tbody>
                {rows_html}
            </tbody>
        </table>

        {conflict_html}

        <p style="margin-top: 25px; font-size: 12px; color: #95a5a6; text-align: center;">
            Generated autonomously by NEXUS Assistant.
        </p>
    </body>
    </html>
    """

    # 4. Dispatch Email via SMTP
    msg = MIMEMultipart("alternative")
    msg["Subject"] = f"⚡ NEXUS Daily Plan: {date_context}"
    msg["From"] = sender_email
    msg["To"] = recipient_email
    msg.attach(MIMEText(html_content, "html"))

    try:
        with smtplib.SMTP("smtp.gmail.com", 587) as server:
            server.starttls()
            server.login(sender_email, sender_password)
            server.sendmail(sender_email, recipient_email, msg.as_string())
        print(f"📧 Notification successfully dispatched to {recipient_email}!")
        return True
    except Exception as e:
        print(f"❌ Failed to dispatch email: {e}")
        return False

In [ ]:
# Cell 9: NEXUS Complete Pipeline
def run_nexus_pipeline(user_prompt: str, recipient_email: Optional[str] = None):
    # Use current system time as the context baseline
    now_str = datetime.now().strftime("%Y-%m-%d %H:%M")

    print("🤖 [1/4] NEXUS is analyzing your input with Gemini...")
    parsed = parse_user_schedule_intent(user_prompt, now_str)

    print("💾 [2/4] Saving tasks to persistent SQLite database...")
    save_tasks_to_db(parsed.tasks)

    print("⚙️ [3/4] Running deterministic scheduling algorithm...")
    schedule_res = generate_optimized_schedule(parsed, now_str)
    save_schedule_to_db(schedule_res.schedule, now_str)

    print("📊 [4/4] Current Tasks in Database:")
    display(get_tasks_as_dataframe())

    # Send email if address provided
    if recipient_email:
        send_nexus_schedule_email(recipient_email, schedule_res, now_str)

    return schedule_res

In [ ]:
# Cell 10: Run Full Pipeline
sample_input = """
I need to finish my Java DSA assignment by 5 PM tomorrow.
I also have a DBMS quiz tomorrow morning.
I want to spend 2 hours on my college project.
I have college classes from 9 AM to 4 PM.
"""

result = run_nexus_pipeline(sample_input, recipient_email=None)

🤖 [1/4] NEXUS is analyzing your input with Gemini...
💾 [2/4] Saving tasks to persistent SQLite database...
⚙️ [3/4] Running deterministic scheduling algorithm...
📊 [4/4] Current Tasks in Database:


,id,title,priority,duration_minutes,deadline,status
0,1,Java DSA Assignment,HIGH,120,2026-10-10 17:00,PENDING
1,2,Prepare for DBMS Quiz,HIGH,60,2026-10-10 09:00,PENDING
2,3,College Project Work,MEDIUM,120,None,PENDING


In [ ]:
# Cell 11: Print the Generated Daily Timetable
from tabulate import tabulate

timeline_rows = []
for slot in result.schedule:
    badge = "📅" if slot.slot_type == "FIXED" else ("☕" if slot.slot_type == "BREAK" else "⚡")
    timeline_rows.append([
        f"{badge} {slot.title}",
        slot.start_time,
        slot.end_time,
        f"{slot.duration_minutes}m",
        slot.details
    ])

print("=" * 70)
print("                     NEXUS DAILY TIMELINE")
print("=" * 70)
print(tabulate(timeline_rows, headers=["Activity", "Start", "End", "Duration", "Type / Notes"], tablefmt="fancy_grid"))

print(f"\n🎯 NEXT ACTION: {result.next_action_recommendation}")
print(f"⏱️ TOTAL FOCUSED STUDY: {result.total_study_minutes // 60}h {result.total_study_minutes % 60}m")

if result.conflicts:
    print("\n⚠️ CONFLICTS DETECTED:")
    for c in result.conflicts:
        print(f"• {c}")
else:
    print("\n✅ All tasks fit into available hours without conflicts.")

                     NEXUS DAILY TIMELINE
╒══════════════════════════╤═════════╤═══════╤════════════╤═══════════════════════════════════╕
│ Activity                 │ Start   │ End   │ Duration   │ Type / Notes                      │
╞══════════════════════════╪═════════╪═══════╪════════════╪═══════════════════════════════════╡
│ ⚡ Prepare for DBMS Quiz │ 07:00   │ 08:00 │ 60m        │ Priority: HIGH                    │
├──────────────────────────┼─────────┼───────┼────────────┼───────────────────────────────────┤
│ ☕ Rest & Re-energize    │ 08:00   │ 08:15 │ 15m        │ Short mental recovery break       │
├──────────────────────────┼─────────┼───────┼────────────┼───────────────────────────────────┤
│ ⚡ Java DSA Assignment   │ 08:15   │ 10:15 │ 120m       │ Priority: HIGH                    │
├──────────────────────────┼─────────┼───────┼────────────┼───────────────────────────────────┤
│ 📅 College Classes       │ 09:00   │ 16:00 │ 420m       │ Fixed Commitment (Non-negotiable) │
├─

In [ ]:
# Cell 12: Dispatch Schedule via SMTP
from google.colab import userdata

# Send to yourself
my_email = userdata.get('SENDER_EMAIL')
now_str = datetime.now().strftime("%Y-%m-%d %H:%M")

print(f"📤 Dispatching schedule briefing to: {my_email}...")
success = send_nexus_schedule_email(my_email, result, now_str)

if success:
    print("✅ Check your inbox! Your daily briefing email has arrived.")
else:
    print("❌ Check your app password credentials in Colab secrets.")

📤 Dispatching schedule briefing to: deelaksha22@gmail.com...
📧 Notification successfully dispatched to deelaksha22@gmail.com!
✅ Check your inbox! Your daily briefing email has arrived.


In [ ]:
# ==============================================================
# NEXUS AI AGENT: COMPLETE ALL-IN-ONE PRODUCTION RUNTIME
# ==============================================================

# 1. Imports & Core Types
import os
import time
import sqlite3
import smtplib
import traceback
from datetime import datetime, timedelta
from typing import List, Optional
from enum import Enum

import pandas as pd
from pydantic import BaseModel, Field
import gradio as gr
from google import genai
from google.genai import types
from google.genai.errors import ServerError, ClientError
from google.colab import userdata

# --------------------------------------------------------------
# 2. Pydantic Schemas
# --------------------------------------------------------------
class PriorityLevel(str, Enum):
    LOW = "LOW"
    MEDIUM = "MEDIUM"
    HIGH = "HIGH"
    CRITICAL = "CRITICAL"

class ExtractedTask(BaseModel):
    title: str = Field(description="Short, clear name of the task")
    description: Optional[str] = Field(default="", description="Relevant details or context")
    deadline: Optional[str] = Field(description="Deadline ISO format YYYY-MM-DD HH:MM, or relative time")
    duration_minutes: int = Field(default=60, description="Estimated duration in minutes")
    priority: PriorityLevel = Field(default=PriorityLevel.MEDIUM, description="Task urgency and importance")
    dependencies: List[str] = Field(default_factory=list, description="Pre-requisite tasks")

class FixedCommitment(BaseModel):
    title: str = Field(description="Name of fixed event (e.g. Classes, Exam, Meeting)")
    start_time: str = Field(description="Start time (HH:MM or YYYY-MM-DD HH:MM)")
    end_time: str = Field(description="End time (HH:MM or YYYY-MM-DD HH:MM)")

class ParsedUserInput(BaseModel):
    tasks: List[ExtractedTask] = Field(description="List of flexible actionable tasks")
    fixed_commitments: List[FixedCommitment] = Field(description="Fixed non-negotiable blocks of time")
    wake_up_time: str = Field(default="08:00", description="User's starting hour (HH:MM)")
    sleep_time: str = Field(default="23:00", description="User's end of day cutoff (HH:MM)")
    missing_information_questions: List[str] = Field(default_factory=list)

class ScheduledSlot(BaseModel):
    title: str
    slot_type: str  # "TASK", "FIXED", or "BREAK"
    start_time: str
    end_time: str
    duration_minutes: int
    details: Optional[str] = ""

class ScheduleResult(BaseModel):
    schedule: List[ScheduledSlot]
    conflicts: List[str]
    next_action_recommendation: Optional[str]
    total_study_minutes: int

# --------------------------------------------------------------
# 3. Database Layer (SQLite)
# --------------------------------------------------------------
DB_FILE = "nexus_tasks.db"

def init_db():
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS tasks (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT NOT NULL,
        description TEXT,
        deadline TEXT,
        duration_minutes INTEGER,
        priority TEXT,
        status TEXT DEFAULT 'PENDING',
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS schedules (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        date_str TEXT,
        slot_type TEXT,
        title TEXT,
        start_time TEXT,
        end_time TEXT,
        duration_minutes INTEGER,
        details TEXT
    )
    """)
    conn.commit()
    conn.close()

def save_tasks_to_db(tasks: List[ExtractedTask]):
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    for task in tasks:
        cursor.execute("""
        INSERT INTO tasks (title, description, deadline, duration_minutes, priority, status)
        VALUES (?, ?, ?, ?, ?, 'PENDING')
        """, (task.title, task.description, task.deadline, task.duration_minutes, task.priority.value))
    conn.commit()
    conn.close()

def save_schedule_to_db(schedule: List[ScheduledSlot], date_str: str):
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM schedules WHERE date_str = ?", (date_str,))
    for slot in schedule:
        cursor.execute("""
        INSERT INTO schedules (date_str, slot_type, title, start_time, end_time, duration_minutes, details)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (date_str, slot.slot_type, slot.title, slot.start_time, slot.end_time, slot.duration_minutes, slot.details))
    conn.commit()
    conn.close()

def update_task_status(task_id: int, new_status: str):
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("UPDATE tasks SET status = ? WHERE id = ?", (new_status.upper(), task_id))
    conn.commit()
    conn.close()

def get_tasks_as_dataframe() -> pd.DataFrame:
    conn = sqlite3.connect(DB_FILE)
    df = pd.read_sql_query("SELECT id, title, priority, duration_minutes, deadline, status FROM tasks", conn)
    conn.close()
    return df

# Initialize DB on load
init_db()

# --------------------------------------------------------------
# 4. Cognitive Parser (Gemini API)
# --------------------------------------------------------------
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)
MODEL_ID = "gemini-3.8-flash"

def parse_user_schedule_intent(user_raw_text: str, current_context_time: str) -> ParsedUserInput:
    system_instruction = f"""
    You are the Cognitive Input Engine for NEXUS.
    Current Context: {current_context_time}

    Rules:
    1. Extract negotiable tasks and non-negotiable commitments (classes, exams, shifts).
    2. Default standard exam/class duration to 60-120 minutes if unspecified.
    3. Output valid JSON matching the schema strictly.
    """
    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=user_raw_text,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    response_mime_type="application/json",
                    response_schema=ParsedUserInput,
                    temperature=0.1
                )
            )
            return ParsedUserInput.model_validate_json(response.text)
        except ServerError:
            time.sleep((attempt + 1) * 2)
        except Exception as e:
            if attempt == 2:
                raise e
            time.sleep(2)

# --------------------------------------------------------------
# 5. Deterministic Scheduling Engine
# --------------------------------------------------------------
def parse_time_helper(time_str: str, base_date: datetime) -> datetime:
    try:
        if len(time_str) == 5 and ":" in time_str:
            hour, minute = map(int, time_str.split(":"))
            return base_date.replace(hour=hour, minute=minute, second=0, microsecond=0)
        return datetime.fromisoformat(time_str)
    except Exception:
        return base_date + timedelta(hours=1)

def generate_optimized_schedule(parsed_input: ParsedUserInput, current_date_str: str) -> ScheduleResult:
    base_date = datetime.strptime(current_date_str, "%Y-%m-%d %H:%M")
    day_start = parse_time_helper(parsed_input.wake_up_time, base_date)
    day_end = parse_time_helper(parsed_input.sleep_time, base_date)

    fixed_blocks = []
    for fc in parsed_input.fixed_commitments:
        f_start = parse_time_helper(fc.start_time, base_date)
        f_end = parse_time_helper(fc.end_time, base_date)
        fixed_blocks.append({"title": fc.title, "start": f_start, "end": f_end})
    fixed_blocks.sort(key=lambda x: x["start"])

    free_intervals = []
    cursor = day_start
    for block in fixed_blocks:
        if block["start"] > cursor:
            free_intervals.append({"start": cursor, "end": block["start"]})
        cursor = max(cursor, block["end"])
    if cursor < day_end:
        free_intervals.append({"start": cursor, "end": day_end})

    priority_weights = {
        PriorityLevel.CRITICAL: 4,
        PriorityLevel.HIGH: 3,
        PriorityLevel.MEDIUM: 2,
        PriorityLevel.LOW: 1
    }

    def task_sort_key(task: ExtractedTask):
        dl_score = 999999
        if task.deadline:
            try:
                dl_dt = parse_time_helper(task.deadline, base_date)
                dl_score = (dl_dt - base_date).total_seconds()
            except Exception:
                pass
        return (dl_score, -priority_weights.get(task.priority, 1))

    sorted_tasks = sorted(parsed_input.tasks, key=task_sort_key)
    final_timeline: List[ScheduledSlot] = []
    conflicts: List[str] = []
    total_study_minutes = 0

    for b in fixed_blocks:
        dur = int((b["end"] - b["start"]).total_seconds() // 60)
        final_timeline.append(ScheduledSlot(
            title=b["title"],
            slot_type="FIXED",
            start_time=b["start"].strftime("%H:%M"),
            end_time=b["end"].strftime("%H:%M"),
            duration_minutes=dur,
            details="Fixed Commitment (Non-negotiable)"
        ))

    for task in sorted_tasks:
        remaining_duration = task.duration_minutes
        placed = False
        for interval in free_intervals:
            interval_dur = int((interval["end"] - interval["start"]).total_seconds() // 60)
            if interval_dur >= remaining_duration:
                task_start = interval["start"]
                task_end = task_start + timedelta(minutes=remaining_duration)

                if task.deadline:
                    try:
                        dl_dt = parse_time_helper(task.deadline, base_date)
                        if task_end > dl_dt:
                            conflicts.append(f"CONFLICT: '{task.title}' finishes past its deadline {task.deadline}!")
                    except Exception:
                        pass

                final_timeline.append(ScheduledSlot(
                    title=task.title,
                    slot_type="TASK",
                    start_time=task_start.strftime("%H:%M"),
                    end_time=task_end.strftime("%H:%M"),
                    duration_minutes=remaining_duration,
                    details=f"Priority: {task.priority.value}"
                ))
                total_study_minutes += remaining_duration

                break_dur = 15 if remaining_duration >= 45 else 0
                new_start = task_end + timedelta(minutes=break_dur)
                if break_dur > 0 and new_start < interval["end"]:
                    final_timeline.append(ScheduledSlot(
                        title="Rest & Re-energize",
                        slot_type="BREAK",
                        start_time=task_end.strftime("%H:%M"),
                        end_time=new_start.strftime("%H:%M"),
                        duration_minutes=break_dur,
                        details="Short mental recovery break"
                    ))
                interval["start"] = new_start
                placed = True
                break

        if not placed:
            conflicts.append(f"UNSCHEDULED: '{task.title}' ({task.duration_minutes}m) could not fit into free hours.")

    final_timeline.sort(key=lambda s: s.start_time)
    next_task = next((s for s in final_timeline if s.slot_type == "TASK"), None)
    recommendation = f"Start with '{next_task.title}' at {next_task.start_time}." if next_task else "No immediate pending tasks for today!"

    return ScheduleResult(
        schedule=final_timeline,
        conflicts=conflicts,
        next_action_recommendation=recommendation,
        total_study_minutes=total_study_minutes
    )

# --------------------------------------------------------------
# 6. Priority Email Module (SMTP)
# --------------------------------------------------------------
def send_priority_email_alert(recipient_email: str, parsed_data: ParsedUserInput, schedule_res: ScheduleResult) -> tuple[bool, str]:
    try:
        sender_email = userdata.get('SENDER_EMAIL').strip()
        sender_pass = userdata.get('SENDER_APP_PASSWORD').strip().replace(" ", "")
    except Exception as e:
        return False, f"Missing Colab Secrets: {e}"

    priority_order = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3}
    sorted_tasks = sorted(parsed_data.tasks, key=lambda t: priority_order.get(t.priority.value, 4))
    badge_colors = {"CRITICAL": "#d9534f", "HIGH": "#f0ad4e", "MEDIUM": "#0275d8", "LOW": "#5cb85c"}

    task_rows_html = "".join([
        f"""<tr style="border-bottom: 1px solid #ddd;">
            <td style="padding: 8px;"><span style="background-color: {badge_colors.get(t.priority.value, '#6c757d')}; color: white; padding: 2px 6px; border-radius: 4px; font-weight: bold; font-size: 11px;">{t.priority.value}</span></td>
            <td style="padding: 8px;"><strong>{t.title}</strong></td>
            <td style="padding: 8px;">{t.duration_minutes}m</td>
            <td style="padding: 8px; color: #b00;"><strong>{t.deadline or 'Flexible'}</strong></td>
        </tr>""" for t in sorted_tasks
    ])

    sched_rows_html = "".join([
        f"""<tr style="border-bottom: 1px solid #eee;">
            <td style="padding: 8px; font-weight: bold;">{s.start_time} - {s.end_time}</td>
            <td style="padding: 8px;">{s.title}</td>
            <td style="padding: 8px;">{s.duration_minutes}m</td>
        </tr>""" for s in schedule_res.schedule
    ])

    html_body = f"""
    <html>
    <body style="font-family: Arial, sans-serif; max-width: 650px; margin: auto; padding: 15px;">
        <div style="background-color: #2c3e50; color: white; padding: 15px; border-radius: 6px;">
            <h2 style="margin: 0;">⚡ NEXUS Priority Action Plan</h2>
            <small>Updated: {datetime.now().strftime('%A, %d %b %Y %I:%M %p')}</small>
        </div>
        <div style="background-color: #eaf2f8; padding: 12px; margin: 15px 0; border-radius: 4px;">
            <strong>🎯 Next Action:</strong> {schedule_res.next_action_recommendation}
        </div>
        <h3>🔥 Tasks Ranked by Priority</h3>
        <table style="width: 100%; border-collapse: collapse;">{task_rows_html}</table>
        <h3>📅 Daily Timeline</h3>
        <table style="width: 100%; border-collapse: collapse;">{sched_rows_html}</table>
    </body>
    </html>
    """

    msg = MIMEMultipart("alternative")
    msg["Subject"] = f"⚡ NEXUS Daily Briefing ({datetime.now().strftime('%b %d')})"
    msg["From"] = sender_email
    msg["To"] = recipient_email
    msg.attach(MIMEText(html_body, "html"))

    try:
        with smtplib.SMTP("smtp.gmail.com", 587) as server:
            server.starttls()
            server.login(sender_email, sender_pass)
            server.sendmail(sender_email, recipient_email, msg.as_string())
        return True, "Email successfully sent!"
    except Exception as e:
        return False, f"SMTP Error: {e}"

# --------------------------------------------------------------
# 7. Rescheduling & Gradio Application
# --------------------------------------------------------------
def recalculate_active_schedule():
    conn = sqlite3.connect(DB_FILE)
    df_pending = pd.read_sql_query(
        "SELECT title, description, deadline, duration_minutes, priority FROM tasks WHERE status = 'PENDING'",
        conn
    )
    conn.close()

    tasks_list = []
    for _, row in df_pending.iterrows():
        tasks_list.append(ExtractedTask(
            title=row["title"],
            description=row["description"] or "",
            deadline=row["deadline"],
            duration_minutes=int(row["duration_minutes"]),
            priority=PriorityLevel(row["priority"])
        ))

    now_context = datetime.now().strftime("%Y-%m-%d %H:%M")
    recalc_input = ParsedUserInput(
        tasks=tasks_list,
        fixed_commitments=[],
        wake_up_time="08:00",
        sleep_time="23:00"
    )
    return generate_optimized_schedule(recalc_input, now_context)

def handle_create_schedule(user_input: str, send_email: bool):
    if not user_input.strip():
        return "⚠️ Please type your task description first.", [], get_tasks_as_dataframe()

    now_context = datetime.now().strftime("%Y-%m-%d %H:%M")
    try:
        parsed = parse_user_schedule_intent(user_input, now_context)
        save_tasks_to_db(parsed.tasks)

        sched_res = generate_optimized_schedule(parsed, now_context)
        save_schedule_to_db(sched_res.schedule, now_context)

        email_status = ""
        if send_email:
            try:
                my_email = userdata.get('SENDER_EMAIL')
                success, msg = send_priority_email_alert(my_email, parsed, sched_res)
                email_status = f"\n📧 {msg}"
            except Exception as e:
                email_status = f"\n⚠️ Email failed: {e}"

        timeline_rows = [
            [s.start_time, s.end_time, s.title, f"{s.duration_minutes}m", s.slot_type, s.details]
            for s in sched_res.schedule
        ]

        status = f"🎯 Next Focus: {sched_res.next_action_recommendation}{email_status}\n\n"
        if sched_res.conflicts:
            status += "⚠️ Conflicts:\n" + "\n".join([f"• {c}" for c in sched_res.conflicts])
        else:
            status += "✅ Schedule built without conflicts."

        return status, timeline_rows, get_tasks_as_dataframe()
    except Exception as err:
        return f"❌ Error: {str(err)}", [], get_tasks_as_dataframe()

def handle_mark_complete(task_id: int):
    if task_id is None:
        return get_tasks_as_dataframe(), [], "⚠️ Please enter a valid Task ID."

    update_task_status(int(task_id), "COMPLETED")
    sched_res = recalculate_active_schedule()

    timeline_rows = [
        [s.start_time, s.end_time, s.title, f"{s.duration_minutes}m", s.slot_type, s.details]
        for s in sched_res.schedule
    ]
    return get_tasks_as_dataframe(), timeline_rows, f"✅ Task #{int(task_id)} completed! Schedule recalculated."

# Launch Gradio
with gr.Blocks(title="NEXUS AI Agent") as final_app:
    gr.Markdown("# ⚡ NEXUS: Autonomous Task Planning Agent")
    gr.Markdown("Agentic productivity system with persistent storage, dynamic rescheduling, and priority alerts.")

    with gr.Row():
        with gr.Column(scale=2):
            task_input = gr.Textbox(
                label="Describe your day & tasks",
                placeholder="e.g., I have an exam at 11 AM. Study 2 hours for DBMS, complete Java homework by 6 PM...",
                lines=4
            )
            email_toggle = gr.Checkbox(label="Send priority email digest to my inbox", value=True)
            submit_btn = gr.Button("⚡ Generate & Prioritize Schedule", variant="primary")
        with gr.Column(scale=1):
            status_box = gr.Textbox(label="Agent Status & Email Alerts", lines=6, interactive=False)

    gr.Markdown("### 🕒 Optimized Daily Timeline")
    timeline_table = gr.Dataframe(
        headers=["Start", "End", "Activity", "Duration", "Type", "Details"],
        interactive=False
    )

    gr.Markdown("### 💾 SQLite Task Manager & Dynamic Rescheduling")
    task_table = gr.Dataframe(value=get_tasks_as_dataframe(), interactive=False)

    with gr.Row():
        selected_id = gr.Number(label="Task ID", precision=0)
        complete_btn = gr.Button("✅ Mark as Completed & Recalculate", variant="secondary")
        feedback_display = gr.Textbox(label="Action Feedback", interactive=False)

    submit_btn.click(
        fn=handle_create_schedule,
        inputs=[task_input, email_toggle],
        outputs=[status_box, timeline_table, task_table]
    )

    complete_btn.click(
        fn=handle_mark_complete,
        inputs=[selected_id],
        outputs=[task_table, timeline_table, feedback_display]
    )

final_app.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://64b29315538689d5f0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ==============================================================================
# NEXUS AI AGENT — SINGLE-CELL PRODUCTION RUNTIME
# ==============================================================================

# 1. Core Imports
import os
import re
import time
import sqlite3
import smtplib
import traceback
from datetime import datetime, timedelta
from typing import List, Optional
from enum import Enum

import pandas as pd
from pydantic import BaseModel, Field
import gradio as gr

from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

from google import genai
from google.genai import types
from google.genai.errors import ServerError, ClientError
from google.colab import userdata

# ------------------------------------------------------------------------------
# 2. Data Schemas (Pydantic)
# ------------------------------------------------------------------------------
class PriorityLevel(str, Enum):
    LOW = "LOW"
    MEDIUM = "MEDIUM"
    HIGH = "HIGH"
    CRITICAL = "CRITICAL"

class ExtractedTask(BaseModel):
    title: str = Field(default="Task", description="Name of the task")
    description: Optional[str] = Field(default="", description="Task details")
    deadline: Optional[str] = Field(default=None, description="Deadline format")
    duration_minutes: int = Field(default=60, description="Duration in minutes")
    priority: PriorityLevel = Field(default=PriorityLevel.MEDIUM, description="Urgency")
    dependencies: List[str] = Field(default_factory=list)

class FixedCommitment(BaseModel):
    title: str = Field(default="Fixed Event", description="Name of event")
    start_time: str = Field(default="09:00", description="Start time")
    end_time: str = Field(default="11:00", description="End time")

class ParsedUserInput(BaseModel):
    tasks: List[ExtractedTask] = Field(default_factory=list)
    fixed_commitments: List[FixedCommitment] = Field(default_factory=list)
    wake_up_time: str = Field(default="08:00")
    sleep_time: str = Field(default="23:00")

class ScheduledSlot(BaseModel):
    title: str
    slot_type: str  # "TASK", "FIXED", or "BREAK"
    start_time: str
    end_time: str
    duration_minutes: int
    details: Optional[str] = ""

class ScheduleResult(BaseModel):
    schedule: List[ScheduledSlot]
    conflicts: List[str]
    next_action_recommendation: Optional[str]
    total_study_minutes: int

# ------------------------------------------------------------------------------
# 3. Database Layer (SQLite)
# ------------------------------------------------------------------------------
DB_FILE = "nexus_tasks.db"

def init_db():
    conn = sqlite3.connect(DB_FILE)
    c = conn.cursor()
    c.execute("""
    CREATE TABLE IF NOT EXISTS tasks (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT NOT NULL,
        description TEXT,
        deadline TEXT,
        duration_minutes INTEGER,
        priority TEXT,
        status TEXT DEFAULT 'PENDING',
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)
    c.execute("""
    CREATE TABLE IF NOT EXISTS schedules (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        date_str TEXT,
        slot_type TEXT,
        title TEXT,
        start_time TEXT,
        end_time TEXT,
        duration_minutes INTEGER,
        details TEXT
    )
    """)
    conn.commit()
    conn.close()

def save_tasks_to_db(tasks: List[ExtractedTask]):
    if not tasks:
        return
    conn = sqlite3.connect(DB_FILE)
    c = conn.cursor()
    for t in tasks:
        c.execute("""
        INSERT INTO tasks (title, description, deadline, duration_minutes, priority, status)
        VALUES (?, ?, ?, ?, ?, 'PENDING')
        """, (t.title, t.description, str(t.deadline or 'None'), t.duration_minutes, t.priority.value))
    conn.commit()
    conn.close()

def save_schedule_to_db(schedule: List[ScheduledSlot], date_str: str):
    conn = sqlite3.connect(DB_FILE)
    c = conn.cursor()
    c.execute("DELETE FROM schedules WHERE date_str = ?", (date_str,))
    for s in schedule:
        c.execute("""
        INSERT INTO schedules (date_str, slot_type, title, start_time, end_time, duration_minutes, details)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (date_str, s.slot_type, s.title, s.start_time, s.end_time, s.duration_minutes, s.details))
    conn.commit()
    conn.close()

def update_task_status(task_id: int, new_status: str):
    conn = sqlite3.connect(DB_FILE)
    c = conn.cursor()
    c.execute("UPDATE tasks SET status = ? WHERE id = ?", (new_status.upper(), task_id))
    conn.commit()
    conn.close()

def get_tasks_as_dataframe() -> pd.DataFrame:
    conn = sqlite3.connect(DB_FILE)
    df = pd.read_sql_query("SELECT id, title, priority, duration_minutes, deadline, status FROM tasks", conn)
    conn.close()
    return df

init_db()

# ------------------------------------------------------------------------------
# 4. Cognitive Parser (Gemini API with Local Fallback)
# ------------------------------------------------------------------------------
try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)
except Exception:
    client = None

def fallback_local_parse(text: str) -> ParsedUserInput:
    """Local rule-based fallback when Gemini API hits 429 quota exhaustion."""
    tasks = []
    fixed = []

    # Split text into sentences / segments
    sentences = re.split(r'[\n\.\,\;]+', text)
    for seg in sentences:
        s = seg.strip()
        if not s:
            continue

        # Check for fixed events
        if any(w in s.lower() for w in ["exam", "class", "meeting", "interview", "quiz"]):
            time_match = re.search(r'(\d{1,2}(?::\d{2})?\s*(?:am|pm)?)', s, re.IGNORECASE)
            start_t = "09:00"
            if time_match:
                raw_t = time_match.group(1).lower().replace(" ", "")
                if "pm" in raw_t:
                    num = int(re.findall(r'\d+', raw_t)[0])
                    start_t = f"{num+12 if num != 12 else 12:02d}:00"
                elif "am" in raw_t:
                    num = int(re.findall(r'\d+', raw_t)[0])
                    start_t = f"{num if num != 12 else 0:02d}:00"

            end_hour = (int(start_t.split(":")[0]) + 2) % 24
            fixed.append(FixedCommitment(
                title=s.title(),
                start_time=start_t,
                end_time=f"{end_hour:02d}:00"
            ))
        else:
            # Flexible task
            dur = 60
            dur_match = re.search(r'(\d+)\s*(?:hour|hr|hrs)', s, re.IGNORECASE)
            if dur_match:
                dur = int(dur_match.group(1)) * 60
            else:
                min_match = re.search(r'(\d+)\s*(?:min|minute|mins)', s, re.IGNORECASE)
                if min_match:
                    dur = int(min_match.group(1))

            prio = PriorityLevel.HIGH if any(w in s.lower() for w in ["assignment", "urgent", "submit", "project"]) else PriorityLevel.MEDIUM
            tasks.append(ExtractedTask(
                title=s.capitalize(),
                duration_minutes=dur,
                priority=prio,
                deadline="Tomorrow" if "tomorrow" in s.lower() else "This Week"
            ))

    if not tasks and not fixed:
        tasks.append(ExtractedTask(title="Study & Review Agenda", duration_minutes=60, priority=PriorityLevel.MEDIUM))

    return ParsedUserInput(tasks=tasks, fixed_commitments=fixed)

def parse_user_schedule_intent(user_raw_text: str, current_context_time: str) -> tuple[ParsedUserInput, str]:
    system_instruction = f"""
    You are the Cognitive Input Engine for NEXUS. Context: {current_context_time}
    Separate flexible tasks from fixed commitments (exams, meetings, classes).
    Infer durations (60-120 mins) and priorities. Output strict JSON matching schema.
    """

    if client:
        try:
            response = client.models.generate_content(
                model="gemini-3.8-flash",
                contents=user_raw_text,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    response_mime_type="application/json",
                    response_schema=ParsedUserInput,
                    temperature=0.1
                )
            )
            return ParsedUserInput.model_validate_json(response.text), "Gemini AI Engine (Online)"
        except Exception as e:
            print(f"⚠️ Switching to fallback parser due to: {e}")

    # Use fallback parser if API key is invalid or quota is exhausted (429)
    return fallback_local_parse(user_raw_text), "Local Fallback Engine (API Quota Limit Met)"

# --------------------------------------------------------------
# 5. Deterministic Scheduling Engine
# --------------------------------------------------------------
def parse_time_helper(time_str: str, base_date: datetime) -> datetime:
    try:
        clean = time_str.strip()
        if len(clean) == 5 and ":" in clean:
            h, m = map(int, clean.split(":"))
            return base_date.replace(hour=h, minute=m, second=0, microsecond=0)
        return datetime.fromisoformat(clean[:16])
    except Exception:
        return base_date + timedelta(hours=1)

def generate_optimized_schedule(parsed_input: ParsedUserInput, current_date_str: str) -> ScheduleResult:
    base_date = datetime.strptime(current_date_str, "%Y-%m-%d %H:%M")
    day_start = parse_time_helper(parsed_input.wake_up_time, base_date)
    day_end = parse_time_helper(parsed_input.sleep_time, base_date)

    fixed_blocks = []
    for fc in parsed_input.fixed_commitments:
        f_start = parse_time_helper(fc.start_time, base_date)
        f_end = parse_time_helper(fc.end_time, base_date)
        if f_end <= f_start:
            f_end = f_start + timedelta(hours=2)
        fixed_blocks.append({"title": fc.title, "start": f_start, "end": f_end})
    fixed_blocks.sort(key=lambda x: x["start"])

    free_intervals = []
    cursor = day_start
    for block in fixed_blocks:
        if block["start"] > cursor:
            free_intervals.append({"start": cursor, "end": block["start"]})
        cursor = max(cursor, block["end"])
    if cursor < day_end:
        free_intervals.append({"start": cursor, "end": day_end})

    priority_weights = {
        PriorityLevel.CRITICAL: 4,
        PriorityLevel.HIGH: 3,
        PriorityLevel.MEDIUM: 2,
        PriorityLevel.LOW: 1
    }

    sorted_tasks = sorted(parsed_input.tasks, key=lambda t: -priority_weights.get(t.priority, 1))
    final_timeline: List[ScheduledSlot] = []
    conflicts: List[str] = []
    total_study_minutes = 0

    for b in fixed_blocks:
        dur = int((b["end"] - b["start"]).total_seconds() // 60)
        final_timeline.append(ScheduledSlot(
            title=b["title"],
            slot_type="FIXED",
            start_time=b["start"].strftime("%H:%M"),
            end_time=b["end"].strftime("%H:%M"),
            duration_minutes=dur,
            details="Fixed Commitment (Non-negotiable)"
        ))

    for task in sorted_tasks:
        remaining_duration = task.duration_minutes
        placed = False
        for interval in free_intervals:
            interval_dur = int((interval["end"] - interval["start"]).total_seconds() // 60)
            if interval_dur >= remaining_duration:
                task_start = interval["start"]
                task_end = task_start + timedelta(minutes=remaining_duration)

                final_timeline.append(ScheduledSlot(
                    title=task.title,
                    slot_type="TASK",
                    start_time=task_start.strftime("%H:%M"),
                    end_time=task_end.strftime("%H:%M"),
                    duration_minutes=remaining_duration,
                    details=f"Priority: {task.priority.value}"
                ))
                total_study_minutes += remaining_duration

                break_dur = 15 if remaining_duration >= 45 else 0
                new_start = task_end + timedelta(minutes=break_dur)
                if break_dur > 0 and new_start < interval["end"]:
                    final_timeline.append(ScheduledSlot(
                        title="Rest & Re-energize",
                        slot_type="BREAK",
                        start_time=task_end.strftime("%H:%M"),
                        end_time=new_start.strftime("%H:%M"),
                        duration_minutes=break_dur,
                        details="Short mental rest break"
                    ))
                interval["start"] = new_start
                placed = True
                break

        if not placed:
            conflicts.append(f"UNSCHEDULED: '{task.title}' ({task.duration_minutes}m) could not fit today.")

    final_timeline.sort(key=lambda s: s.start_time)
    next_task = next((s for s in final_timeline if s.slot_type == "TASK"), None)
    rec = f"Start with '{next_task.title}' at {next_task.start_time}." if next_task else "No pending flexible tasks for today!"

    return ScheduleResult(
        schedule=final_timeline,
        conflicts=conflicts,
        next_action_recommendation=rec,
        total_study_minutes=total_study_minutes
    )

# --------------------------------------------------------------
# 6. Priority Email Module (SMTP)
# --------------------------------------------------------------
def send_priority_email_alert(recipient_email: str, parsed_data: ParsedUserInput, schedule_res: ScheduleResult) -> tuple[bool, str]:
    try:
        sender_email = userdata.get('SENDER_EMAIL').strip()
        sender_pass = userdata.get('SENDER_APP_PASSWORD').strip().replace(" ", "")
    except Exception as e:
        return False, f"Email secrets missing or invalid: {e}"

    priority_order = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3}
    sorted_tasks = sorted(parsed_data.tasks, key=lambda t: priority_order.get(t.priority.value, 4))
    badge_colors = {"CRITICAL": "#d9534f", "HIGH": "#f0ad4e", "MEDIUM": "#0275d8", "LOW": "#5cb85c"}

    task_rows_html = "".join([
        f"""<tr style="border-bottom: 1px solid #ddd;">
            <td style="padding: 8px;"><span style="background-color: {badge_colors.get(t.priority.value, '#6c757d')}; color: white; padding: 2px 6px; border-radius: 4px; font-weight: bold; font-size: 11px;">{t.priority.value}</span></td>
            <td style="padding: 8px;"><strong>{t.title}</strong></td>
            <td style="padding: 8px;">{t.duration_minutes}m</td>
            <td style="padding: 8px; color: #b00;"><strong>{t.deadline or 'Flexible'}</strong></td>
        </tr>""" for t in sorted_tasks
    ])

    sched_rows_html = "".join([
        f"""<tr style="border-bottom: 1px solid #eee;">
            <td style="padding: 8px; font-weight: bold;">{s.start_time} - {s.end_time}</td>
            <td style="padding: 8px;">{s.title}</td>
            <td style="padding: 8px;">{s.duration_minutes}m</td>
        </tr>""" for s in schedule_res.schedule
    ])

    html_body = f"""
    <html>
    <body style="font-family: Arial, sans-serif; max-width: 650px; margin: auto; padding: 15px;">
        <div style="background-color: #2c3e50; color: white; padding: 15px; border-radius: 6px;">
            <h2 style="margin: 0;">⚡ NEXUS Priority Action Plan</h2>
            <small>Updated: {datetime.now().strftime('%A, %d %b %Y %I:%M %p')}</small>
        </div>
        <div style="background-color: #eaf2f8; padding: 12px; margin: 15px 0; border-radius: 4px;">
            <strong>🎯 Next Action:</strong> {schedule_res.next_action_recommendation}
        </div>
        <h3>🔥 Tasks Ranked by Priority</h3>
        <table style="width: 100%; border-collapse: collapse;">{task_rows_html or '<tr><td>No flexible tasks recorded.</td></tr>'}</table>
        <h3>📅 Daily Timeline</h3>
        <table style="width: 100%; border-collapse: collapse;">{sched_rows_html or '<tr><td>No active slots.</td></tr>'}</table>
    </body>
    </html>
    """

    msg = MIMEMultipart("alternative")
    msg["Subject"] = f"⚡ NEXUS Daily Briefing ({datetime.now().strftime('%b %d')})"
    msg["From"] = sender_email
    msg["To"] = recipient_email
    msg.attach(MIMEText(html_body, "html"))

    try:
        with smtplib.SMTP("smtp.gmail.com", 587) as server:
            server.starttls()
            server.login(sender_email, sender_pass)
            server.sendmail(sender_email, recipient_email, msg.as_string())
        return True, "Email successfully sent!"
    except Exception as e:
        return False, f"SMTP Error: {e}"

# --------------------------------------------------------------
# 7. Dynamic Rescheduling & Gradio Application
# --------------------------------------------------------------
def recalculate_active_schedule():
    conn = sqlite3.connect(DB_FILE)
    df_pending = pd.read_sql_query(
        "SELECT title, description, deadline, duration_minutes, priority FROM tasks WHERE status = 'PENDING'",
        conn
    )
    conn.close()

    tasks_list = []
    for _, row in df_pending.iterrows():
        tasks_list.append(ExtractedTask(
            title=row["title"],
            description=row["description"] or "",
            deadline=row["deadline"],
            duration_minutes=int(row["duration_minutes"]),
            priority=PriorityLevel(row["priority"])
        ))

    now_context = datetime.now().strftime("%Y-%m-%d %H:%M")
    recalc_input = ParsedUserInput(
        tasks=tasks_list,
        fixed_commitments=[],
        wake_up_time="08:00",
        sleep_time="23:00"
    )
    return generate_optimized_schedule(recalc_input, now_context)

def handle_create_schedule(user_input: str, send_email: bool):
    if not user_input.strip():
        return "⚠️ Please type your task description first.", [], get_tasks_as_dataframe()

    now_context = datetime.now().strftime("%Y-%m-%d %H:%M")
    try:
        parsed, engine_used = parse_user_schedule_intent(user_input, now_context)
        save_tasks_to_db(parsed.tasks)

        sched_res = generate_optimized_schedule(parsed, now_context)
        save_schedule_to_db(sched_res.schedule, now_context)

        email_status = ""
        if send_email:
            try:
                my_email = userdata.get('SENDER_EMAIL')
                success, msg = send_priority_email_alert(my_email, parsed, sched_res)
                email_status = f"\n📧 {msg}"
            except Exception as e:
                email_status = f"\n⚠️ Email failed: {e}"

        timeline_rows = [
            [s.start_time, s.end_time, s.title, f"{s.duration_minutes}m", s.slot_type, s.details]
            for s in sched_res.schedule
        ]

        status = f"⚙️ Engine: {engine_used}\n🎯 Next Focus: {sched_res.next_action_recommendation}{email_status}\n\n"
        if sched_res.conflicts:
            status += "⚠️ Conflicts:\n" + "\n".join([f"• {c}" for c in sched_res.conflicts])
        else:
            status += "✅ Schedule built without conflicts."

        return status, timeline_rows, get_tasks_as_dataframe()
    except Exception as err:
        return f"❌ Error: {str(err)}", [], get_tasks_as_dataframe()

def handle_mark_complete(task_id: int):
    if task_id is None:
        return get_tasks_as_dataframe(), [], "⚠️ Please enter a valid Task ID."

    update_task_status(int(task_id), "COMPLETED")
    sched_res = recalculate_active_schedule()

    timeline_rows = [
        [s.start_time, s.end_time, s.title, f"{s.duration_minutes}m", s.slot_type, s.details]
        for s in sched_res.schedule
    ]
    return get_tasks_as_dataframe(), timeline_rows, f"✅ Task #{int(task_id)} completed! Schedule recalculated."

# Launch the Gradio Interface
with gr.Blocks(title="NEXUS AI Agent") as final_app:
    gr.Markdown("# ⚡ NEXUS: Autonomous Task Planning Agent")
    gr.Markdown("Agentic productivity system with persistent storage, dynamic rescheduling, and priority email alerts.")

    with gr.Row():
        with gr.Column(scale=2):
            task_input = gr.Textbox(
                label="Describe your day & tasks",
                placeholder="i have a exam for Monday and study for java for 2 hours and also i have to submit Computer Networks assignment for tomorrow",
                lines=4
            )
            email_toggle = gr.Checkbox(label="Send priority email digest to my inbox", value=True)
            submit_btn = gr.Button("⚡ Generate & Prioritize Schedule", variant="primary")
        with gr.Column(scale=1):
            status_box = gr.Textbox(label="Agent Status & Email Alerts", lines=6, interactive=False)

    gr.Markdown("### 🕒 Optimized Daily Timeline")
    timeline_table = gr.Dataframe(
        headers=["Start", "End", "Activity", "Duration", "Type", "Details"],
        interactive=False
    )

    gr.Markdown("### 💾 SQLite Task Manager & Dynamic Rescheduling")
    task_table = gr.Dataframe(value=get_tasks_as_dataframe(), interactive=False)

    with gr.Row():
        selected_id = gr.Number(label="Task ID", precision=0)
        complete_btn = gr.Button("✅ Mark as Completed & Recalculate", variant="secondary")
        feedback_display = gr.Textbox(label="Action Feedback", interactive=False)

    submit_btn.click(
        fn=handle_create_schedule,
        inputs=[task_input, email_toggle],
        outputs=[status_box, timeline_table, task_table]
    )

    complete_btn.click(
        fn=handle_mark_complete,
        inputs=[selected_id],
        outputs=[task_table, timeline_table, feedback_display]
    )

final_app.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://65289198b9b6d2b872.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


⚠️ Switching to fallback parser due to: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 8h4m4.953520754s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global',